# Notebook 05: Out-of-Sample Validation

This notebook evaluates the continuous HMM (with and without jumps) on held-out 2025 data,
reproducing Figure 4 and the OoS portion of Table 2.

### Key Question
Does the IS-calibrated model generalize to unseen data?

### Outputs
- **Figure 4**: KS p-values (IS/OoS), density fan chart, ACF for jump paths
- **Table 2 (OoS)**: Full metrics table for out-of-sample window

## Setup

In [ ]:
include("../Include.jl");

In [ ]:
# --- CONSTANTS ---
ticker = "SPY";
number_of_states = 13;
number_of_paths = 1000;
L = 252;
risk_free_rate = 0.0;
Δt = 1/252;

## Load Tuned Models and OoS Data

In [ ]:
# --- LOAD TUNED MODEL ---
model_path = joinpath(_PATH_TO_DATA, "CHMM-$(ticker)-K$(number_of_states)-tuned.jld2");

if isfile(model_path)
    saved = load(model_path);
    model = saved["model"];
    jump_model = saved["jump_model"];
    T_matrix = saved["T_matrix"];
    π_stationary = saved["pi_stationary"];
    in_sample_data = saved["in_sample_data"];
    best_ε = saved["best_epsilon"];
    best_λ = saved["best_lambda"];
    println("Loaded tuned model. ε*=$best_ε, λ*=$best_λ")
else
    error("Run Notebooks 02-03 first.")
end

K = length(model.states);
start_distribution = Categorical(π_stationary);

In [ ]:
# --- LOAD OOS DATA ---
oos_dataset_raw = MyOutOfSamplePortfolioDataSet() |> x -> x["dataset"];
R_oos = log_growth_matrix(oos_dataset_raw, ticker; Δt=Δt, risk_free_rate=risk_free_rate);
oos_steps = length(R_oos);

println("OoS data: $(oos_steps) observations for $ticker")

## Simulate OoS-Length Paths

In [ ]:
# --- SIMULATE OoS-LENGTH PATHS ---
oos_decoded_nj = Array{Float64,2}(undef, oos_steps, number_of_paths);
oos_decoded_wj = Array{Float64,2}(undef, oos_steps, number_of_paths);

for i in 1:number_of_paths
    # CHMM-NJ
    start_state = rand(start_distribution);
    states_nj = model(start_state, oos_steps);
    for j in 1:oos_steps
        oos_decoded_nj[j, i] = rand(model.emission[states_nj[j]]);
    end
    
    # CHMM-WJ
    start_state = rand(start_distribution);
    states_wj = jump_model(start_state, oos_steps);
    for j in 1:oos_steps
        oos_decoded_wj[j, i] = rand(jump_model.emission[states_wj[j]]);
    end
end

println("OoS simulation complete: $(number_of_paths) paths x $(oos_steps) steps.")

## OoS Validation Metrics

In [ ]:
# --- OoS METRICS ---
function compute_oos_metrics(observed, simulated_archive; α=0.05, label="Model")
    n_paths = size(simulated_archive, 2);
    n_obs = length(observed);
    μ_obs = mean(observed); σ_obs = std(observed);
    kurt_obs = sum(((observed .- μ_obs) ./ σ_obs).^4) / n_obs - 3.0;
    
    ks_pass = 0; kurt_sum = 0.0; w1_sum = 0.0; hell_sum = 0.0;
    ks_pvals = Float64[];
    
    for i in 1:n_paths
        sim = simulated_archive[:, i];
        pval = pvalue(ApproximateTwoSampleKSTest(observed, sim));
        push!(ks_pvals, pval);
        if pval > α; ks_pass += 1; end
        
        μ_s = mean(sim); σ_s = std(sim);
        kurt_sum += sum(((sim .- μ_s) ./ σ_s).^4) / length(sim) - 3.0;
        
        obs_sorted = sort(observed); sim_sorted = sort(sim);
        n_min = min(length(obs_sorted), length(sim_sorted));
        obs_q = [obs_sorted[round(Int, k*length(obs_sorted)/n_min)] for k in 1:n_min];
        sim_q = [sim_sorted[round(Int, k*length(sim_sorted)/n_min)] for k in 1:n_min];
        w1_sum += mean(abs.(obs_q .- sim_q));
        
        edges = range(minimum([observed; sim])-10, maximum([observed; sim])+10, length=101);
        h_obs = fit(Histogram, observed, edges).weights ./ length(observed);
        h_sim = fit(Histogram, sim, edges).weights ./ length(sim);
        hell_sum += sqrt(sum((sqrt.(h_obs) .- sqrt.(h_sim)).^2)) / sqrt(2);
    end
    
    println("\n--- $label (OoS, $(n_paths) paths) ---")
    println("KS pass rate:    $(round(100*ks_pass/n_paths, digits=1))%")
    println("Excess kurtosis: $(round(kurt_sum/n_paths, digits=2)) (observed: $(round(kurt_obs, digits=2)))")
    println("Wasserstein-1:   $(round(w1_sum/n_paths, digits=3))")
    println("Hellinger:       $(round(hell_sum/n_paths, digits=4))")
    
    return (ks_rate=100*ks_pass/n_paths, kurtosis=kurt_sum/n_paths,
            wasserstein=w1_sum/n_paths, hellinger=hell_sum/n_paths, ks_pvals=ks_pvals)
end

oos_nj = compute_oos_metrics(R_oos, oos_decoded_nj; label="CHMM-NJ");
oos_wj = compute_oos_metrics(R_oos, oos_decoded_wj; label="CHMM-WJ");

## Figure 4: Out-of-Sample Validation

In [ ]:
# --- FIGURE 4a: OoS KS p-values ---
p4a = histogram(oos_wj.ks_pvals, bins=50, normalize=true, alpha=0.6, color=:navy,
    label="CHMM-WJ", title="(a) OoS KS p-values", titlefontsize=9,
    xlabel="p-value", ylabel="Density");
vline!(p4a, [0.05], lw=2, color=:red, ls=:dash, label="α=0.05");

# --- FIGURE 4b: OoS Density Fan Chart ---
x_grid = range(minimum(R_oos)-50, maximum(R_oos)+50, length=500);

p4b = plot(title="(b) OoS Density Fan Chart", titlefontsize=9,
    xlabel="Excess Growth Rate", ylabel="Density");

# Plot a sample of simulated densities
for i in 1:min(50, number_of_paths)
    density!(p4b, oos_decoded_wj[:, i], color=:navy, alpha=0.05, label="");
end
density!(p4b, R_oos, lw=3, color=:red, label="Observed OoS");

# --- FIGURE 4c: OoS ACF comparison ---
τ_oos = 1:min(L, oos_steps-1);
acf_oos_obs = autocor(abs.(R_oos), τ_oos);

# Compute ACF envelope from simulated paths
acf_oos_archive = hcat([autocor(abs.(oos_decoded_wj[:, i]), τ_oos) for i in 1:min(200, number_of_paths)]...);
acf_oos_mean = mean(acf_oos_archive, dims=2)[:];
acf_oos_p10 = [quantile(acf_oos_archive[t, :], 0.10) for t in 1:length(τ_oos)];
acf_oos_p90 = [quantile(acf_oos_archive[t, :], 0.90) for t in 1:length(τ_oos)];

p4c = plot(τ_oos, acf_oos_obs, lw=2, color=:red, ls=:dash, label="Observed OoS",
    title="(c) OoS ACF(|Gₜ|)", titlefontsize=9,
    xlabel="Lag (trading days)", ylabel="ACF(|Gₜ|)");
plot!(p4c, τ_oos, acf_oos_mean, lw=2, color=:navy, label="CHMM-WJ (mean)");
plot!(p4c, τ_oos, acf_oos_p10, fillrange=acf_oos_p90, alpha=0.2, color=:navy, label="10-90th pctl");

# Combine
fig4 = plot(p4a, p4b, p4c, layout=(1, 3), size=(1400, 400),
    plot_title="Figure 4: Out-of-Sample Validation — $(ticker) (2025)",
    plot_titlefontsize=12);
display(fig4)

savefig(fig4, joinpath(_PATH_TO_FIGURES, "Fig-4-OoS-Validation-$(ticker).svg"))
savefig(fig4, joinpath(_PATH_TO_FIGURES, "Fig-4-OoS-Validation-$(ticker).pdf"))

## Summary: IS vs OoS Comparison Table

In [ ]:
# --- SUMMARY TABLE ---
println("\n" * "="^70)
println("Table 2: In-Sample vs Out-of-Sample Comparison — $ticker")
println("="^70)
println("                    | CHMM-NJ (IS) | CHMM-WJ (IS) | CHMM-NJ (OoS) | CHMM-WJ (OoS)")
println("-"^70)
println("KS pass rate (%)   |  Run NB 04    |  Run NB 04    | $(round(oos_nj.ks_rate,digits=1))           | $(round(oos_wj.ks_rate,digits=1))")
println("Excess kurtosis    |               |               | $(round(oos_nj.kurtosis,digits=2))          | $(round(oos_wj.kurtosis,digits=2))")
println("Wasserstein-1      |               |               | $(round(oos_nj.wasserstein,digits=3))        | $(round(oos_wj.wasserstein,digits=3))")
println("Hellinger          |               |               | $(round(oos_nj.hellinger,digits=4))       | $(round(oos_wj.hellinger,digits=4))")
println("="^70)
println("\nNote: Run Notebook 04 for full IS metrics. Combine results for the complete Table 2.")

## Disclaimer

This content is offered solely for research and educational purposes. It does not constitute financial advice.